In [2]:
"""
Описание задания (кратко)
Задача: Обучить TimeGAN на истории индекса S&P 500 (SPY) для генерации синтетических дневных цен (OHLCV). 
Сравнить статистику реальных и синтетических рядов. Использовать модель для создания альтернативных 
сценариев после кризисного мартовского блока 2020 года (стресс-тест).

Шаги:

Загрузить SPY за 2015–2023 (yfinance), масштабировать признаки в [0,1], нарезать на окна длиной 24 дня.

Обучить TimeGAN (5000 эпох).

Сгенерировать синтетические последовательности, обратно масштабировать.

Визуально сравнить реальные/синтетические цены, распределения доходностей, автокорреляции.

Оценить качество через TSTR: обучить линейную регрессию на синтетических доходностях → предсказать реальные → сравнить MAE с baseline (предсказание нуля).

Сгенерировать 5 альтернативных сценариев, стартующих с реального блока марта 2020 (условная генерация).

Написать выводы: можно ли применять синтетику для банковских стресс-тестов.

Данные: SPY (Open, High, Low, Close, Volume) за 2015–2023.

Ожидаемый результат: Jupyter Notebook с выполненными TODO, графиками, таблицей статистик, оценкой TSTR и стресс-сценариями
"""

'\nОписание задания (кратко)\nЗадача: Обучить TimeGAN на истории индекса S&P 500 (SPY) для генерации синтетических дневных цен (OHLCV). \nСравнить статистику реальных и синтетических рядов. Использовать модель для создания альтернативных \nсценариев после кризисного мартовского блока 2020 года (стресс-тест).\n\nШаги:\n\nЗагрузить SPY за 2015–2023 (yfinance), масштабировать признаки в [0,1], нарезать на окна длиной 24 дня.\n\nОбучить TimeGAN (5000 эпох).\n\nСгенерировать синтетические последовательности, обратно масштабировать.\n\nВизуально сравнить реальные/синтетические цены, распределения доходностей, автокорреляции.\n\nОценить качество через TSTR: обучить линейную регрессию на синтетических доходностях → предсказать реальные → сравнить MAE с baseline (предсказание нуля).\n\nСгенерировать 5 альтернативных сценариев, стартующих с реального блока марта 2020 (условная генерация).\n\nНаписать выводы: можно ли применять синтетику для банковских стресс-тестов.\n\nДанные: SPY (Open, High, L

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
import yfinance as yf

# Импорт TimeGAN из библиотеки ydata-synthetic
from ydata_synthetic.synthesizers import ModelParameters, TrainParameters
from ydata_synthetic.synthesizers.timeseries import TimeSeriesSynthesizer
from ydata_synthetic.preprocessing.timeseries import processed_stock

# Подавление предупреждений TensorFlow (опционально)
import tensorflow as tf
tf.get_logger().setLevel('ERROR')

# ========================
# 1. ЗАГРУЗКА ДАННЫХ
# ========================
print("Загрузка данных S&P 500 (SPY)...")
ticker = "SPY"
df = yf.download(ticker, start="2015-01-01", end="2023-12-31", progress=False)
df = df[['Open', 'High', 'Low', 'Close', 'Volume']]  # возьмем стандартные OHLCV столбцы
df.reset_index(inplace=True)

print(f"Загружено строк: {len(df)}")
print(f"Пропуски: {df.isnull().sum().sum()}")

# Сохраним даты отдельно для визуализации
dates = df['Date']
df = df.drop(columns=['Date'])

# Проверка на пропуски
# TODO 1.1: Если есть пропуски, заполните их методом forward fill (ffill)
df = # <ВАШ_КОД>

# ========================
# 2. ПРЕДОБРАБОТКА ДАННЫХ
# ========================
print("\n--- Предобработка данных ---")

# TimeGAN ожидает данные в формате: (numpy array [samples, features])
# Все признаки должны быть приведены к шкале [0, 1] с помощью MinMaxScaler
# TODO 2.1: Создайте MinMaxScaler, обучите его на df, преобразуйте данные
scaler = MinMaxScaler()
data_scaled = # <ВАШ_КОД>

# TimeGAN использует сущность 'processed_stock' для резки данных на окна
# TODO 2.2: Задайте длину последовательности (окна). Seq_len = 24 (1 торговый месяц)
# Эта длина должна быть достаточной, чтобы модель уловила временны́е зависимости.
seq_len = 24
processed_data = processed_stock(data_scaled, seq_len=seq_len)

# processed_data — список массивов.
# Обучаем TimeGAN на всех данных, кроме последнего полного окна (как запас)
train_data = processed_data[:-1]

# ========================
# 3. ОПРЕДЕЛЕНИЕ ПАРАМЕТРОВ МОДЕЛИ
# ========================
print("\n--- Определение параметров TimeGAN ---")
# Архитектура TimeGAN: 
# 1) Embedder — RNN, отображает реальные временны́е ряды в скрытое представление.
# 2) Generator — RNN, генерирует синтетические данные из шума.
# 3) Discriminator — RNN, отличает реальные ряды от синтетических.
# 4) Recovery — RNN, восстанавливает из скрытого представления исходные данные.
# Все RNN компоненты имеют одинаковую архитектуру и размерности.

# Основные гиперпараметры (для stock data)
noise_dim = 32      # Размерность входного шума
dim = 128           # Размерность скрытых состояний RNN
batch_size = 128
lr = 5e-4           # Learning rate

# Параметры модели
gan_args = ModelParameters(batch_size=batch_size,
                           lr=lr,
                           noise_dim=noise_dim,
                           layers_dim=dim,
                           latent_dim=24,
                           gamma=1)

train_args = TrainParameters(epochs=5000,         # Для семинара можно 5000, в продакшене 50000+
                             sequence_length=seq_len,
                             number_sequences=len(train_data))

print(f"Размерность шума: {noise_dim}")
print(f"Размерность скрытого состояния: {dim}")
print(f"Epochs: {train_args.epochs}")

# ========================
# 4. ОБУЧЕНИЕ TimeGAN
# ========================
print("\n--- Обучение TimeGAN ---")
# TODO 4.1: Создайте экземпляр TimeSeriesSynthesizer с параметрами:
# modelname='timegan', model_parameters=gan_args
synth = # <ВАШ_КОД>

# Если обучение занимает слишком долго (особенно на CPU), его можно остановить досрочно,
# модель все равно сгенерирует правдоподобные данные.

# TODO 4.2: Обучите модель (метод .fit), передав train_data и train_args
# Для ускорения на семинаре можно уменьшить epochs в train_args
synth.fit(train_data, train_args)
# После обучения модель можно сохранить: synth.save('timegan_spy.pkl')

# ========================
# 5. ГЕНЕРАЦИЯ СИНТЕТИЧЕСКИХ ДАННЫХ
# ========================
print("\n--- Генерация синтетических данных ---")
# TODO 5.1: Сгенерируйте столько же синтетических последовательностей, сколько было в train_data
# Используйте метод .sample(n_samples=...)
n_samples = len(train_data)
synth_data_blocks = synth.sample(n_samples)

# TODO 5.2: Соберите сгенерированные данные обратно в единый DataFrame в исходном масштабе.
# Обратите внимание: synth_data_blocks — это список, каждый элемент — np.array формы (seq_len, n_features)
# Нужно объединить их в один длинный ряд. Результат сохраните в переменную 'synth_df'.
synth_df_list = []
for block in synth_data_blocks:
    synth_df_list.append(pd.DataFrame(block, columns=df.columns))
synth_df_scaled = pd.concat(synth_df_list, ignore_index=True)

# TODO 5.3: Примените обратное масштабирование inverse_transform, чтобы вернуть реальные цены
synth_df = pd.DataFrame(scaler.inverse_transform(synth_df_scaled), columns=df.columns)

# ========================
# 6. ВИЗУАЛИЗАЦИЯ И СРАВНЕНИЕ РАСПРЕДЕЛЕНИЙ
# ========================
print("\n--- Сравнение оригинальных и синтетических данных ---")

# График 1: Сравнение динамики цены Close (первые 500 точек)
plt.figure(figsize=(14, 6))
plt.plot(df['Close'].values[:500], label='Real SPY Close', alpha=0.7)
plt.plot(synth_df['Close'].values[:500], label='Synthetic Close', alpha=0.7)
plt.title('Сравнение временны́х рядов: реальный vs синтетический (Close)')
plt.xlabel('Time steps')
plt.ylabel('Price')
plt.legend()
plt.grid(True)
plt.show()

# TODO 6.1: Постройте распределение дневных доходностей для реальных и синтетических данных.
# Логарифмическая доходность = np.log(price / price.shift(1)). Сравните через гистограммы или KDE plot.
real_returns = # <ВАШ_КОД>
synth_returns = # <ВАШ_КОД>

plt.figure(figsize=(12, 5))
sns.kdeplot(real_returns, label='Real Returns', shade=True)
sns.kdeplot(synth_returns, label='Synthetic Returns', shade=True)
plt.title('Распределение логарифмических доходностей')
plt.xlabel('Daily Return')
plt.legend()
plt.show()

# TODO 6.2: Рассчитайте и сравните ключевые статистики для доходностей:
# mean, std, skewness, kurtosis для реальных и синтетических данных.
real_stats = # <ВАШ_КОД>
synth_stats = # <ВАШ_КОД>
print(pd.DataFrame({'Real': real_stats, 'Synthetic': synth_stats}))

# ========================
# 7. ОЦЕНКА КАЧЕСТВА: ВАЛИДАЦИЯ "TRAIN ON SYNTHETIC, TEST ON REAL" (TSTR)
# ========================
print("\n--- Валидация TSTR (Train on Synthetic, Test on Real) ---")
# Идея: предсказываем будущую доходность на реальных данных, но модель обучаем на синтетических.
# Если синтетические данные хорошо воспроизводят статистику, качество предсказания будет близким.
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# Функция для создания X и y: предсказываем следующую доходность на основе окна прошлых доходностей
def prepare_ts_data(returns, window=5):
    X, y = [], []
    for i in range(window, len(returns)):
        X.append(returns[i-window:i])
        y.append(returns[i])
    return np.array(X), np.array(y)

window=5
X_real, y_real = prepare_ts_data(real_returns, window)
X_synth, y_synth = prepare_ts_data(synth_returns, window)

# TODO 7.1: Обучите линейную регрессию на синтетических данных, сделайте предсказание для реальных.
model = LinearRegression()
model.fit(X_synth, y_synth)
y_pred = model.predict(X_real)

# TODO 7.2: Сравните ошибку предсказания с бенчмарком (предсказание нулем, т.е. всегда 0)
baseline_mae = np.mean(np.abs(y_real))
model_mae = mean_absolute_error(y_real, y_pred)

print(f"MAE baseline (predict 0): {baseline_mae:.6f}")
print(f"MAE TSTR (train on synth): {model_mae:.6f}")

# ========================
# 8. СТРЕСС-ТЕСТИРОВАНИЕ: ГЕНЕРАЦИЯ КРИЗИСНЫХ СЦЕНАРИЕВ
# ========================
print("\n--- Генерация условных сценариев (стресс-тест) ---")
# Условная генерация: генерируем синтетические последовательности,
# которые стартуют с указанного начального состояния (из реальной кризисной точки).

# Выбираем кризисный период: март 2020
covid_idx = dates[(dates >= '2020-03-01') & (dates < '2020-03-31')].index[0]
covid_block = data_scaled[covid_idx:covid_idx + seq_len]

# TODO 8.1: Сгенерируйте 5 сценариев развития после заданного начального блока.
# В ydata-synthetic для условной генерации можно использовать условие start_block.
# Если библиотека не поддерживает start_block, можно обучить Conditional TimeGAN или
# в качестве упрощения: сэмплировать из обученной модели, выбрать последовательности,
# которые начинаются близко к covid_block (по MSE).
# Для семинара оставим простой вариант: генерируем множество последовательностей и отбираем ближайшие.
n_candidates = 100
candidates = synth.sample(n_candidates)
# Вычисляем MSE начала каждой последовательности с covid_block
mse_start = [np.mean((block[:10] - covid_block[:10])**2) for block in candidates]
best_indices = np.argsort(mse_start)[:5]
stress_scenarios = [candidates[i] for i in best_indices]

# Визуализация стресс-сценариев
plt.figure(figsize=(14, 6))
plt.plot(df['Close'].values[covid_idx:covid_idx+seq_len], label='Real COVID crash', linewidth=2)
for i, scenario in enumerate(stress_scenarios):
    scenario_prices = scaler.inverse_transform(scenario)[:, 3]  # Close column index = 3
    plt.plot(scenario_prices, linestyle='--', alpha=0.8, label=f'Scenario {i+1}')
plt.title('Стресс-тест: альтернативные сценарии после марта 2020')
plt.xlabel('Days from start')
plt.ylabel('SPY Price')
plt.legend()
plt.grid(True)
plt.show()

# ========================
# 9. БОНУС: СРАВНЕНИЕ АВТОКОРРЕЛЯЦИИ (МЕТРИКА ДЛЯ ВРЕМЕННЫХ РЯДОВ)
# ========================
from statsmodels.tsa.stattools import acf
# TODO 9.1: Рассчитайте автокорреляционную функцию для реальных и синтетических доходностей.
# Постройте общий график для первых 20 лагов.
real_acf = acf(real_returns, nlags=20)
synth_acf = acf(synth_returns, nlags=20)

plt.figure(figsize=(10, 5))
plt.stem(range(len(real_acf)), real_acf, linefmt='b-', markerfmt='bo', label='Real ACF')
plt.stem(range(len(synth_acf)), synth_acf, linefmt='r-', markerfmt='ro', label='Synthetic ACF')
plt.title('Автокорреляционная функция доходностей')
plt.xlabel('Lag')
plt.ylabel('Autocorrelation')
plt.legend()
plt.grid(True)
plt.show()

# ========================
# ВЫВОДЫ И ВОПРОСЫ ДЛЯ ДИСКУССИИ
# ========================
"""
### Обсуждение результатов:

1. **Качество генерации**: Насколько близки распределения доходностей и ACF реальных и синтетических данных? 
2. **TSTR валидация**: Что говорит метрика MAE о пригодности синтетических данных для обучения моделей?
3. **Стресс-тестирование**: Как полученные сценарии можно применить для оценки рисков портфеля?
4. **Ограничения**: В чем недостатки TimeGAN? (требует много данных, сложность обучения)
5. **Бизнес-применение**: Как банк может использовать генеративные модели для PFE (Probability of Future Events) и ECL (Expected Credit Loss)?
"""